# Fill the missing values of the word forms with the values of the lemmas (only using spaCy)

In [ ]:
import pandas as pd
import os
import re
# import nltk
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
import spacy
from spacy.symbols import ORTH
from multiprocessing import Pool
import time

from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
import numpy as np
from gBatch_lemmatize import lemmatize_batch,lemmatize_step1,lemmatize_step2,lemmatize_step3,lemmatize_step4,lemmatize_spacy
# warnings.filterwarnings("ignore", message="numpy.dtype size changed")
# warnings.filterwarnings("ignore", message="numpy.ufunc size changed")
# nltk.download('wordnet')
# nltk.download('omw-1.4')
# nltk.download('punkt')
# nltk.download('averaged_perceptron_tagger')  # For POS tagging
# Dir_github = '/work/desai-lab/xuanyang/Project/Semantic/analysis/FactorAnalysis/github'
Dir_github = '/work/desai-lab/xuanyang/Project/Semantic/dissemination/github/DiscoFMRI'
repo = 'FactorAnalysis_fMRI'

##### Download the database from SCOPE
You will need to manually download the SCOPE metabase from https://sc.edu/study/colleges_schools/artsandsciences/psychology/research_clinical_facilities/scope/search.php. Select Data -> All -> Download


In [ ]:
# skip if already done
df = pd.read_excel(os.path.join(Dir_github,'data','SCOPE','data_with_metadata.xlsx'))
df2 = pd.read_excel(os.path.join(Dir_github,'data','SCOPE','data_with_metadata.xlsx'),usecols=["Word"],keep_default_na=False, na_values=[])
df['Word'] = df2['Word']
for label in ['Word','NonWord']:
    file = os.path.join(Dir_github,'data',f"SCOPE_{label.lower()}.csv")
    df.loc[df['Status']==label].to_csv(file,index=False)
    

##### Load the SCOPE dataset

In [ ]:
# Load the file downloaded from the SCOPE database, which only contains real words. 
# read the "Word" column separately to avoid a problem caused by some special words like "NA", 'nan'
df_SCOPE_raw = pd.read_csv(os.path.join(Dir_github,'data','SCOPE',f"SCOPE_word.csv"))
df = pd.read_csv(os.path.join(Dir_github,'data','SCOPE',f"SCOPE_word.csv"), usecols=["Word"], keep_default_na=False, na_values=[],encoding="latin-1")
df_SCOPE_raw['Word'] = df['Word'] 
df_SCOPE_raw = df_SCOPE_raw.loc[~df_SCOPE_raw['Word'].duplicated()].reset_index() # the word "used" is duplicated
# # df_SCOPE.drop(columns=['Unnamed: 246'],inplace=True) # there is an extra column contains nothing 
print(f'The original SCOPE database has {df_SCOPE_raw.shape[0]} words, {df_SCOPE_raw.shape[1]-3} variables.')
df_SCOPE_raw

## Lemmatization

Step1: We used the lemmas directly provided by the `Subtlex-UK` database, which employs Stanford NLP parsing based on the most frequent part-of-speech (PoS) tags. 61,842 words in SCOPE were lemmatized in this way.

Step2: For American spellings with available PoS tags provided in the `Subtlex-US` database but not the directly available dominated lemmas, we used the `NLTK` package to extract the corresponding lemmas based on those most frequently used PoS tags. 5,717 words were lemmatized in this way. 

Step3: For the remaining words (N = 38,433) that do not have available PoS tags, we used the natural language processing package `spaCy` to automatically extract the lemma of every raw word form. 

Step4: This is an additional step to assign the raw word forms as the lemmas if they are special cases, such as cannot, gonna, wanna, and special conpounds containing "-" and other punctuations.

In [ ]:
def lemmatize_step3only(df_SCOPE,nlp):
    onset = time.time()
    with Pool(processes=48) as pool:  
        # results = pool.map(do_process, [row for _, row in df_spacy.iterrows()])
        results = pool.starmap(lemmatize_spacy, [(i,row,nlp) for i, row in df_SCOPE.iterrows()])
    
    
    df_spacy = pd.concat(results, ignore_index=True)
    # df_SCOPE.loc[:,['Lemma_spacy','PoS_tag_spacy']] = df_final[['Lemma_spacy','PoS_tag_spacy']]
    
    df_SCOPE = df_SCOPE.set_index('Word').join(df_spacy.set_index('Word')[['Lemma_spacy','PoS_tag_spacy']], how='left').reset_index()
    
    df_SCOPE['Lemma'] = df_SCOPE['Lemma_UKUS']
    df_SCOPE['PoS_tag'] = df_SCOPE['PoS_UKUS']
    df_SCOPE.loc[~df_SCOPE['Lemma_spacy'].isna(),'Lemma'] = df_SCOPE.loc[~df_SCOPE['Lemma_spacy'].isna(),'Lemma_spacy'] 
    df_SCOPE.loc[~df_SCOPE['PoS_tag_spacy'].isna(),'PoS_tag'] = df_SCOPE.loc[~df_SCOPE['PoS_tag_spacy'].isna(),'PoS_tag_spacy']
    offset = time.time()
    print(f"Step3:\n---> {round(offset-onset,2)}s")
    return(df_SCOPE)



nlp = spacy.load('en_core_web_sm')
nlp.max_length = 1013000
# Add specials case to keep as single tokens
List_punct = ["'","_","-","."]

df_SCOPE = df_SCOPE_raw[['Word','DPoS_VanH']].copy()
special_cases = list(df_SCOPE.loc[df_SCOPE['Word'].apply(lambda x: any(punct in x for punct in List_punct)),'Word'].values)
special_cases = special_cases+["cant","couldnt","wont","gonna","wanna","kinda",
                              # "woulda","coulda","shoulda","outta","sorta","oughta","dunno",
                              ]
special_cases = special_cases + ['cannot', 'gotta', 'hes', 'id', 'Id', 'shes', 'thats', 'theres',
       'theyre', 'wed', 'whats', 'whos', 'whys']

for case in special_cases:
    nlp.tokenizer.add_special_case(case, [{ORTH: case}])

df_spacyonly = lemmatize_step3only(df_SCOPE,nlp)

In [ ]:
df_lemmas = lemmatize_step4(df_spacyonly,special_cases)

In [ ]:

df_lemmas.to_csv(os.path.join(Dir_github,'data','SCOPE','SCOPE_lemmatization_output_spacyonly.csv'),index=False)

In [3]:
df_lemmas = lemmatize_batch(])

Step1:
---> 61842/105992 (58.35%) words have available lemmas.
---> 33.54s
Step2:
---> 67559/105992 (63.74%) words have available lemmas.
---> 60.66s
Step3:
---> 62.43s
Step4:

---> 2108 special cases.
---> 0.13s


Check lemmatization quality

In [5]:
df1 = df_lemmas.set_index('Word')
df_uniqueLemma = pd.DataFrame(df1['Lemma'].unique(),columns=['Lemma']).set_index('Lemma')
print(f'{df_uniqueLemma.shape[0]} unique lemmas.')

df_uniqueLemma['inSCOPE'] = 0
df2 = df_lemmas.loc[df_lemmas['Word']==df_lemmas['Lemma']].set_index('Word')

df_uniqueLemma.loc[df2.index] = 1
print(f"{df_uniqueLemma.loc[df_uniqueLemma['inSCOPE']==0].shape[0]} lemmas cannot be found in the Word column.")
df_uniqueLemma.loc[df_uniqueLemma['inSCOPE']==0].head(50).index

75227 unique lemmas.
4302 lemmas cannot be found in the Word column.


Index(['aa', 'abashe', 'abberation', 'abc', 'abdominis', 'abelson',
       'abernathy', 'ablard', 'abra', 'abroade', 'absentia', 'abyssinian',
       'accessor', 'accomodation', 'accompani', 'achille', 'acidifie',
       'acorde', 'acquiesence', 'acropoli', 'adame', 'adamo', 'adams',
       'adamson', 'addlebraine', 'aderholds', 'adio', 'admixe', 'adoni',
       'adrar', 'adrian', 'adrianople', 'adrien', 'adventist', 'adventuresse',
       'aegean', 'aerobacter', 'aerodonetic', 'aeromechanic',
       'aerotherapeutic', 'aeschbacher', 'aeschylus', 'affor', 'afforest',
       'afghani', 'afghanistani', 'afranio', 'afrikaan', 'afrikaner',
       'afterburne'],
      dtype='object', name='Lemma')

In [6]:
df_SCOPE = df_SCOPE_raw.set_index('Word').join(df_lemmas.set_index('Word')[['Lemma','PoS_tag']],how='left').reset_index()
df_SCOPE['idx'] = df_SCOPE.index

List_var_LemmaScore = [x for x in df_SCOPE.columns if (x.startswith('Freq_') or x.startswith('CD_'))]
for var in List_var_LemmaScore:
    if 'Zipf' not in var:
        df_SCOPE[f'{var}_expo'] = 10**df_SCOPE[var]
    else:
        df_SCOPE[f'{var}_expo'] = 10**(df_SCOPE[var]-3)
df_SCOPE

,Word,index,Freq_HAL,Freq_KF,Freq_SUBTLEXUS,Freq_SUBTLEXUS_Zipf,Freq_SUBTLEXUK,Freq_SUBTLEXUK_Zipf,Freq_Blog,Freq_Twitter,...,Freq_CobW_expo,Freq_CobS_expo,Freq_Cob_Lemmas_expo,Freq_CobW_Lemmas_expo,Freq_CobS_Lemmas_expo,CD_SUBTLEXUS_expo,CD_SUBTLEXUK_expo,CD_Blog_expo,CD_Twitter_expo,CD_News_expo
0,'em,0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,22.998526,81.997385,28.183829,22.908677,81.283052,NaN,NaN,NaN,NaN,NaN
1,'neath,1,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.000000,1.000000,1.000000,1.000000,1.000000,NaN,NaN,NaN,NaN,NaN
2,'re,2,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,8.000184,43.003123,10.000000,7.943282,42.657952,NaN,NaN,NaN,NaN,NaN
3,'shun,3,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.000000,1.000000,1.000000,1.000000,1.000000,NaN,NaN,NaN,NaN,NaN
4,'tis,4,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2.999853,4.000369,3.019952,3.019952,3.981072,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105987,zygote,105988,NaN,NaN,0.90309,2.194924,1.041393,1.737128,0.477121,0.778151,...,NaN,NaN,NaN,NaN,NaN,6.0,7.0,3.0,6.0,3.0
105988,zygotic,105989,NaN,NaN,NaN,NaN,0.477121,1.172857,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,3.0,NaN,NaN,NaN
105989,zymogen,105990,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
105990,zymology,105991,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Fill missing values with lemma values, use the lemma-level summed frequency/CD scores

In [33]:
def process_row(word,row,df_SCOPE):
    df1 = df_SCOPE.loc[df_SCOPE['Lemma'] == word].reset_index(drop=True)
    df2 = pd.concat([row.to_frame().T] * df1.shape[0], ignore_index=True)
    
    df3 = df1.combine_first(df2)
    df = df3.copy()
    # Compute Lemma Frequency (the sum scores of all word forms).
    List_var_LemmaScore = [x for x in df_SCOPE.columns if (x.startswith('Freq_') or x.startswith('CD_')) and '_expo' not in x]
    df_sum = df.groupby('Lemma').sum()[[f"{x}_expo" for x in List_var_LemmaScore]]
    for var in List_var_LemmaScore:
        score = df_sum[f"{var}_expo"].values[0]
        if score>0:
            df[f"{var}_LemmaSum"] = np.log10(score)
    return df

def process_in_parallel(row_data):
    word, row = row_data
    return process_row(word,row,df_SCOPE)

onset = time.time()

df_LEMMA = df_SCOPE.copy()
df_LEMMA = df_LEMMA.set_index('Word').loc[df_uniqueLemma.loc[df_uniqueLemma['inSCOPE']==1].index]
rows = list(df_LEMMA.iterrows())
with Pool(processes=48) as pool:
    dfs_tmp = pool.map(process_in_parallel, rows)
dfs_output = pd.concat(dfs_tmp, ignore_index=True)
offset = time.time()
print(f"---> {round(offset-onset,2)}s")


dfs_output.to_csv(os.path.join(Dir_github,'data','SCOPE','SCOPE_lemmatization_output.csv'),index=False)

---> 893.0s


In [60]:
df_LEMMA = df_SCOPE.copy()
dfs_output = pd.read_csv(os.path.join(Dir_github,'data','SCOPE_lemmatization_output.csv'))
dfs_output_text = pd.read_csv(os.path.join(Dir_github,'data','SCOPE_lemmatization_output.csv'),usecols=["Word","Lemma"],keep_default_na=False, na_values=[])
dfs_output[['Word','Lemma']] = dfs_output_text[['Word','Lemma']]

df_LEMMA['flag'] = 0
df_LEMMA.loc[dfs_output.set_index('idx').index,'flag'] = 1
df_SCOPE_lemma = pd.concat([df_LEMMA.loc[df_LEMMA['flag']==0],dfs_output],ignore_index=True)
df_SCOPE_lemma.sort_values(['idx'],inplace=True,ignore_index=True)
df_SCOPE_lemma.to_csv(os.path.join(Dir_github,'data','SCOPE','SCOPE_lemma.csv'),index=False)

df_SCOPE_lemma

,Word,index,Freq_HAL,Freq_KF,Freq_SUBTLEXUS,Freq_SUBTLEXUS_Zipf,Freq_SUBTLEXUK,Freq_SUBTLEXUK_Zipf,Freq_Blog,Freq_Twitter,...,Freq_SUBTLEXUK_LemmaSum,Freq_SUBTLEXUK_Zipf_LemmaSum,Freq_Blog_LemmaSum,Freq_Twitter_LemmaSum,Freq_News_LemmaSum,CD_SUBTLEXUS_LemmaSum,CD_SUBTLEXUK_LemmaSum,CD_Blog_LemmaSum,CD_Twitter_LemmaSum,CD_News_LemmaSum
0,'em,0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,'neath,1,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,'re,2,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,'shun,3,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,'tis,4,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105987,zygote,105988,NaN,NaN,0.90309,2.194924,1.041393,1.737128,0.477121,0.778151,...,1.041393,-1.262872,0.477121,0.778151,0.477121,0.778151,0.845098,0.477121,0.778151,0.477121
105988,zygotic,105989,NaN,NaN,NaN,NaN,0.477121,1.172857,NaN,NaN,...,0.477121,-1.827143,NaN,NaN,NaN,NaN,0.477121,NaN,NaN,NaN
105989,zymogen,105990,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
105990,zymology,105991,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Select variables

We selected 106 variables to maximize the number of words and the number of variables, resulting in 13,850 unique words with available values across all variables.

In [79]:
df_SCOPE_lemma = pd.read_csv(os.path.join(Dir_github,'data','SCOPE','SCOPE_lemma.csv'),low_memory=False)
df_SCOPE_lemma_text = pd.read_csv(os.path.join(Dir_github,'data','SCOPE','SCOPE_lemma.csv'),usecols=["Word","Lemma"],keep_default_na=False, na_values=[])
df_SCOPE_lemma[['Word','Lemma']] = df_SCOPE_lemma_text[['Word','Lemma']]

df_var_113 = pd.read_csv(os.path.join(Dir_github,'data','SCOPE','List_113variables_sorted.csv'))
for var in ['Freq_','CD_']:
    df_var_113.loc[df_var_113['variable'].str.startswith(var),'variable'] = df_var_113.loc[df_var_113['variable'].str.startswith(var),'variable']+'_LemmaSum'

List_var_113 = list(df_var_113['variable'].values)

List_var_Freq_OrthPhon = ['Orth_N_Freq_L', 'Orth_N_Freq_G', 'Orth_N_Freq', 'Orth_N_Freq_L_Mean',
       'Orth_N_Freq_G_Mean', 'Phonographic_N_Freq', 'Phon_N_Freq']
List_106 = list(set(List_var_113)-set(List_var_Freq_OrthPhon))

List_vars = List_106
df_new = df_SCOPE_lemma[['idx','Word','Lemma']+List_vars].dropna()
df_new.to_csv(os.path.join(Dir_github,'data','SCOPE','SCOPE_lemma_var106.csv'),index=False)

df_new

,idx,Word,Lemma,Fam_Brys,OLD20F,OLD20,Mink_Perceptual_Lanc,CD_Blog_LemmaSum,Cumfreq_TASA,UniphonP_St,...,Phon_N,Consistency_Token_FF_C,UnigramF_Avg_U_Log,Sem_N_D,BigramF_Avg_C_Log,Freq_SUBTLEXUK_LemmaSum,CD_SUBTLEXUK_LemmaSum,Haptic_Lanc,Consistency_Token_FB_C,Freqtraj_TASA
35,35,abandon,abandon,0.962963,7.450,2.90,2.744282,3.132260,-0.735424,0.049831,...,0.0,0.987201,5.432017,0.589894,2.645147,3.707911,3.600755,0.294118,0.917246,1.059894
36,36,abandoned,abandon,1.000000,6.329,3.55,3.075033,3.132260,1.035183,0.051145,...,1.0,0.989504,5.457469,0.664088,2.865353,3.707911,3.600755,0.055556,0.793056,0.935349
39,39,abandoning,abandon,0.962963,6.956,3.70,2.744282,3.132260,-1.543549,0.051486,...,0.0,0.991987,5.409765,0.554907,2.697587,3.707911,3.600755,0.294118,0.957437,0.251768
41,41,abandons,abandon,0.962963,7.450,2.90,2.744282,3.132260,-0.735424,0.052438,...,1.0,0.987201,5.430674,0.444684,2.515291,3.707911,3.600755,0.294118,0.917246,1.059894
101,101,abdomen,abdomen,1.000000,6.386,3.00,4.602757,1.954243,-0.401581,0.047912,...,0.0,0.989527,5.421687,0.560194,2.532652,2.759668,2.595496,3.294118,0.939005,0.660592
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105919,105919,zone,zone,1.000000,8.469,1.40,3.091321,3.145196,1.171116,0.042188,...,23.0,1.000000,5.452560,0.654892,3.066132,3.950851,3.682957,0.611111,0.025341,0.642419
105920,105920,zoned,zone,1.000000,7.131,1.90,3.242207,3.145196,1.171116,0.044226,...,9.0,1.000000,5.415118,0.421339,3.040460,3.950851,3.682957,0.485714,0.134066,0.642419
105921,105921,zones,zone,1.000000,7.791,1.65,3.091321,3.145196,-0.383166,0.040259,...,12.0,1.000000,5.446458,0.610629,3.115118,3.950851,3.682957,0.611111,0.248645,1.412152
105922,105922,zoning,zone,1.000000,7.259,2.00,4.005242,3.145196,-1.013596,0.044552,...,7.0,0.999511,5.332065,0.528917,3.295281,3.950851,3.682957,0.875000,0.959106,0.781722
